<a href="https://colab.research.google.com/github/nabinjoshi54/lis5693/blob/main/lab-8/lab_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Lab 8: Topic Modeling Using BERTopic

In this lab, I use the BERTopic package to perform topic modeling on the same dataset that I used in Lab 5. The goal is to explore how BERTopic groups documents into topics and then compare those results with the LDA-based topic modeling from the previous lab. Since BERTopic uses embeddings and clustering instead of only word-frequency patterns, it can capture semantic meaning in a different way.
In Google Colab, it is better to use a GPU for BERTopic because generating embeddings can take more time on CPU.

To enable GPU in Colab:

Runtime > Change runtime type > Hardware accelerator > GPU

Step 1: Install the required packages

First, I install the libraries needed for this lab. BERTopic depends on sentence transformers for embeddings and UMAP/HDBSCAN for clustering.

In [1]:
%%capture
!pip install bertopic sentence-transformers umap-learn hdbscan plotly

Step 2: Import libraries
I import the Python libraries that will be used for loading the dataset, cleaning the text, building the BERTopic model, and visualizing the results.

In [2]:
import pandas as pd
import numpy as np
import re

from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer

Step 3: Load the dataset

I use the same batteries-and-space dataset from Lab 5. This keeps the comparison fair because both LDA and BERTopic are being applied to the same collection of documents.

In [3]:
url = "https://raw.githubusercontent.com/nabinjoshi54/lis5693/main/lab-8/lab5-batteries-and-space.csv"
df = pd.read_csv(url)

df.head()

,Lens ID,Title,Date Published,Publication Year,Publication Type,Source Title,ISSNs,Publisher,Source Country,Author/s,...,PMID,DOI,Microsoft Academic ID,PMCID,Citing Patents Count,References,Citing Works Count,Is Open Access,Open Access License,Open Access Colour
0,000-277-169-899-878,WATER-ACTIVATED DRY-CHARGED LEAD–ACID BATTERIES,NaN,1970.0,book chapter,Research and Development in Non-Mechanical Ele...,NaN,Elsevier,NaN,D.L. Douglas; R.E. Biddick; J.B. Ockerman,...,NaN,10.1016/b978-0-08-013435-2.50011-6,2234449575,NaN,0,026-727-716-648-058; 118-148-506-040-617,3,False,NaN,NaN
1,000-296-458-948-553,Digitally controlled autonomous Li-ion active ...,NaN,2014.0,conference proceedings article,2014 International Conference on Advances in E...,NaN,IEEE,NaN,G Rishivathsala; S Ananda; V. Sreekumar; Nitin...,...,NaN,10.1109/icaecc.2014.7002466,2541639867,NaN,0,029-655-612-502-395; 066-402-048-809-762; 073-...,4,False,NaN,NaN
2,000-395-893-043-269,Space systems � Lithium ion battery for space ...,2016-03-19,2016.0,standard,NaN,NaN,BSI British Standards,NaN,NaN,...,NaN,10.3403/30296384u,NaN,NaN,0,NaN,0,False,NaN,NaN
3,000-550-445-902-514,Cost-Benefit Analysis Model of Single and Hybr...,NaN,2014.0,journal article,Applied Mechanics and Materials,16627482,"Trans Tech Publications, Ltd.",NaN,Yi Feng; Lei Jun Shao; Bang Ling Zhang; Jing Y...,...,NaN,10.4028/www.scientific.net/amm.672-674.503,1999071943,NaN,0,053-147-224-892-52X; 066-432-448-842-155; 095-...,0,False,NaN,NaN
4,000-621-072-737-768,The 2000 NASA Aerospace Battery Workshop,2001-03-01,2001.0,NaN,NaN,NaN,NaN,NaN,Jeffrey C. Brewer,...,NaN,NaN,2801440225,NaN,0,NaN,0,False,NaN,NaN


Step 4: Inspect the data

Before modeling, I check the shape of the dataset and confirm that the abstract column is available because BERTopic will use document text as input.

In [4]:
print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Dataset shape: (1000, 32)

Columns:
['Lens ID', 'Title', 'Date Published', 'Publication Year', 'Publication Type', 'Source Title', 'ISSNs', 'Publisher', 'Source Country', 'Author/s', 'Abstract', 'Volume', 'Issue Number', 'Start Page', 'End Page', 'Fields of Study', 'Keywords', 'MeSH Terms', 'Chemicals', 'Funding', 'Source URLs', 'External URL', 'PMID', 'DOI', 'Microsoft Academic ID', 'PMCID', 'Citing Patents Count', 'References', 'Citing Works Count', 'Is Open Access', 'Open Access License', 'Open Access Colour']


Step 5: Prepare the text data

For topic modeling, I use the `Abstract` column because it contains the main content of each article. I remove missing values, convert everything to string, and do light cleaning so the text is more consistent.

In [5]:
df = df.dropna(subset=["Abstract"]).copy()

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"\s+", " ", text)          # remove extra spaces
    text = re.sub(r"\n", " ", text)           # remove line breaks
    return text.strip()

df["clean_abstract"] = df["Abstract"].apply(clean_text)

documents = df["clean_abstract"].tolist()

print("Number of documents used:", len(documents))
print("\nSample abstract:\n")
print(documents[0][:1000])

Number of documents used: 858

Sample abstract:

the energy storage is required to support spacecraft loads during eclipse and also when the demand is more than power generation at any time. the most widely used energy storage technology is the battery, which stores energy in an electrochemical form. there is a need for highly efficient and long life battery systems for spacecrafts at various power levels. li-ion based battery technology has better performance over traditional battery technologies. the battery system comprises of protection, management and balancing. of these, balancing is most important that ensures the better performance and life of the battery system. a novel energy converter based active/non dissipative cell equalization method is implemented. this method comprises of digitally controlled resonant reset forward converter for individual li-ion cells connected in series. this converter is used to transfer the energy from the higher voltage cell to the lower voltage c

Step 6: Create document embeddings

BERTopic works differently from LDA because it starts by converting each document into an embedding. These embeddings capture semantic meaning, which helps BERTopic group documents based on similarity in meaning rather than only matching word frequencies.

In [6]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Step 7: Build the BERTopic model

Here, I create the BERTopic model. I also define a vectorizer with English stop words removed so the topic words are easier to interpret.

In [7]:
vectorizer_model = CountVectorizer(stop_words="english")

topic_model = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer_model,
    language="english",
    calculate_probabilities=True,
    verbose=True
)

Step 8: Fit the model to the abstracts

Now I run BERTopic on the dataset. The model will generate topic assignments for each document and estimate topic probabilities.

In [8]:
topics, probabilities = topic_model.fit_transform(documents)

2026-04-24 02:52:06,800 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/27 [00:00<?, ?it/s]

2026-04-24 02:53:29,305 - BERTopic - Embedding - Completed ✓
2026-04-24 02:53:29,308 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-24 02:53:43,820 - BERTopic - Dimensionality - Completed ✓
2026-04-24 02:53:43,821 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-24 02:53:43,896 - BERTopic - Cluster - Completed ✓
2026-04-24 02:53:43,902 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-24 02:53:44,035 - BERTopic - Representation - Completed ✓


Step 9: Review the discovered topics

After fitting the model, I inspect the topic summary table. This helps me see how many topics were created, how many documents are assigned to each topic, and which words best represent each topic.

In [9]:
topic_info = topic_model.get_topic_info()
topic_info.head(15)

,Topic,Count,Name,Representation,Representative_Docs
0,-1,291,-1_battery_batteries_power_space,"[battery, batteries, power, space, jats, cells...",[<jats:p>the paper presents methods of providi...
1,0,73,0_model_state_battery_method,"[model, state, battery, method, based, predict...",[prognostics and health management has become ...
2,1,63,1_batteries_jats_lithium_li,"[batteries, jats, lithium, li, electrolyte, su...","[lockheed martin missiles & space (lmms), ultr..."
3,2,54,2_nickel_aerospace_hydrogen_cadmium,"[nickel, aerospace, hydrogen, cadmium, worksho...",[this document contains the proceedings of the...
4,3,51,3_temperature_thermal_battery_heat,"[temperature, thermal, battery, heat, jats, co...","[<jats:p>&lt;div class=""section abstract""&gt;&..."
5,4,45,4_battery_ion_lithium_cell,"[battery, ion, lithium, cell, space, cells, di...",[one goal of the power and propulsion office a...
6,5,44,5_hst_telescope_hubble_battery,"[hst, telescope, hubble, battery, capacity, ba...",[this paper summarizes the hubble space telesc...
7,6,38,6_spacecraft_battery_orbit_performance,"[spacecraft, battery, orbit, performance, goes...",[the upper atmosphere research satellite (uars...
8,7,30,7_ion_satellite_battery_li,"[ion, satellite, battery, li, satellites, desi...",[this paper introduces the lithium ion battery...
9,8,28,8_power_energy_nuclear_space,"[power, energy, nuclear, space, radioisotope, ...",[<jats:p>the increased interest in space activ...


Step 10: Display the top words for several topics

To better understand the model output, I print the representative words from the first few discovered topics.

In [10]:
valid_topics = [t for t in topic_info["Topic"].tolist() if t != -1][:10]

for topic_id in valid_topics:
    print(f"\nTopic {topic_id}:")
    print(topic_model.get_topic(topic_id))


Topic 0:
[('model', np.float64(0.05449009307619632)), ('state', np.float64(0.04838525237627128)), ('battery', np.float64(0.03738763322377012)), ('method', np.float64(0.035120038496576814)), ('based', np.float64(0.03400687045642396)), ('prediction', np.float64(0.03249420050454463)), ('estimation', np.float64(0.031575808147360836)), ('models', np.float64(0.02897721454648584)), ('lithium', np.float64(0.028820972579366656)), ('health', np.float64(0.026649540907378))]

Topic 1:
[('batteries', np.float64(0.04201945246684268)), ('jats', np.float64(0.040595289675668514)), ('lithium', np.float64(0.03697900891389817)), ('li', np.float64(0.035977945025213724)), ('electrolyte', np.float64(0.03540151430345653)), ('sub', np.float64(0.03337793672405038)), ('materials', np.float64(0.03230501800181072)), ('sup', np.float64(0.030381044558591842)), ('ion', np.float64(0.02959740377863959)), ('solid', np.float64(0.02851210824932682))]

Topic 2:
[('nickel', np.float64(0.0824322418729798)), ('aerospace', np

Step 11: Add topic labels back to the dataset

To make the results easier to inspect, I add the assigned topic number to the original dataframe.

In [11]:
df["BERTopic_Topic"] = topics
df[["Title", "BERTopic_Topic"]].head(10)

,Title,BERTopic_Topic
1,Digitally controlled autonomous Li-ion active ...,12
3,Cost-Benefit Analysis Model of Single and Hybr...,-1
4,The 2000 NASA Aerospace Battery Workshop,2
5,Characteristics of Thermal Runaway Propagation...,13
7,The Use of Small Cell Lithium-Ion Batteries fo...,-1
9,A Cosmic Battery,-1
10,Hybrid Electric PropulsionArchitectural Design...,8
11,Improved charging methods for nickel-cadmium b...,6
12,CBERS 3&4 TM Thermal Balance Test Results and ...,-1
13,Rasat Leo Satellite Power System Design and Op...,9


Step 12: Examine a few example documents from selected topics

Looking at example article titles helps confirm whether the topics make sense and whether similar articles are grouped together.

In [12]:
for topic_id in valid_topics[:5]:
    print(f"\n{'='*80}")
    print(f"Examples from Topic {topic_id}")
    print(f"{'='*80}")

    sample_titles = df[df["BERTopic_Topic"] == topic_id]["Title"].head(5).tolist()
    for i, title in enumerate(sample_titles, 1):
        print(f"{i}. {title}")


Examples from Topic 0
1. State of charge estimation and state space model analysis of the Ni-MH battery system
2. Universally-Charging Protocols for Quantum Batteries: A No-Go Theorem
3. An Attenuation Analysis Method for Lithium-ion Battery Based on Multi-core Rlevance Vector Machine
4. A Transformer-CNN Based Transferable Model for State-of-Health Prediction of Lithium-ion Batteries
5. SoC Estimation Lithium Polymer Battery Based on Equivalent Circuit Model and Extended Kalman Filter

Examples from Topic 1
1. Recent developments in electrodes and separators for high-performance lithium–sulfur batteries
2. Design of miniaturized extraction equipment for lithium polymer battery
3. Artificial Solid Electrolyte Interphase Layer for Lithium Metal Anode in High-Energy Lithium Secondary Pouch Cells
4. Lithium titanate as anode material for lithium-ion cells: a review
5. Aerospace lithium solid polymer batteries

Examples from Topic 2
1. The 2000 NASA Aerospace Battery Workshop
2. Nickel-ca

Step 13: Visualize the topic map

This visualization shows how the topics are positioned relative to one another in a reduced-dimensional space. Topics that appear closer together are generally more semantically related.

In [13]:
topic_model.visualize_topics()

Step 14: Visualize topic frequency

This chart shows which topics appear most often in the dataset.

In [14]:
topic_model.visualize_barchart(top_n_topics=10)

Step 15: Visualize the hierarchy of topics

This figure helps show whether some topics can be grouped into broader higher-level themes.

In [15]:
topic_model.visualize_hierarchy()

Task 2: Interpretation and Comparison with Lab 5

In Lab 5, I used LDA and selected the 20-topic model because it had the highest coherence score (0.41). The model identified key themes such as batteries, space systems, thermal behavior, and energy applications. However, many topics had overlapping keywords like "battery", "space", and "power", which made them harder to distinguish.

In Lab 8, BERTopic produced more distinct and meaningful topics. For example, Topic 7 clearly represents battery discharge (battery, lithium, discharge, ion), Topic 6 focuses on spacecraft systems (spacecraft, orbit, launched), and Topic 3 captures thermal behavior (thermal, temperature, heat). These topics are more focused compared to LDA.

The visualizations also show better separation. The topic map shows clearer clustering, and the hierarchical clustering groups related topics such as space systems and battery technologies together, while still keeping them distinct.

LDA relies on word frequency and co-occurrence patterns, which often leads to overlapping topics. On the other hand, BERTopic uses embeddings and clustering to capture semantic meaning, resulting in more distinct and coherent topic separation.

Both methods identified similar themes, but BERTopic produced more coherent and well-separated topics, while LDA showed more overlap between related concepts.

Task 3: Reflection

This lab helped me better understand how different topic modeling approaches work on the same dataset. One thing that went well was that both LDA and BERTopic were able to identify the main themes related to batteries and space applications. BERTopic produced more clear and meaningful topics and made interpretation easier.

One challenge I faced was understanding why LDA topics overlapped so much. The coherence score was higher for the 20-topic model butsome topics were still difficult to distinguish. This showed me that a higher coherence score does not always mean better interpretability. BERTopic also required more time to run because it uses embeddings, which can be computationally expensive.

In my future research, I think embeddings and large language models can be very useful. I found one article published by professor from UC Berkley that did the same work. I will probably do the same work for my research. This can help group similar research papers, identify important trends, and make literature reviews more efficient. I would like to apply these methods to analyze papers in my own research area.